# CVDWesterman

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.CVDWesterman)

class CVDWesterman(LinearReferenceClock):
    def postprocess(self, x):
        """Logistic transform to a CVD risk probability."""
        return torch.sigmoid(x)



In [3]:
model = pya.models.CVDWesterman()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "cvdwesterman"
model.metadata["data_type"] = "DNA methylation"  # Paper: DNA methylation cardiovascular risk score from blood cohorts.
model.metadata["species"] = "Homo sapiens"  # Paper: DNA methylation cardiovascular risk score from blood cohorts.
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Westerman, K. et al. Epigenomic assessment of cardiovascular disease risk and interactions with traditional risk metrics. Journal of the American Heart Association 9, e015299 (2020)."
model.metadata["doi"] = "https://doi.org/10.1161/jaha.119.015299"
model.metadata["notes"] = "Whole-blood DNA-methylation score for cardiovascular risk. The paper's final cross-study learner stacks cohort-specific elastic-net Cox models; the packaged pyaging implementation is a 235-CpG linear score followed by a sigmoid."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: DNA methylation data ... collected using the HumanMethylation450 platform.
model.metadata["predicts"] = ["cardiovascular disease risk"]  # Paper: postprocess_name = 'sigmoid'
model.metadata["training_target"] = ["cardiovascular disease"]  # Paper: CVD events included coronary heart disease, stroke, and death from CVD.
model.metadata["unit"] = ["probability"]  # Paper: Logistic transform to a CVD risk probability.
model.metadata["model_type"] = "elastic net Cox ensemble"  # Paper: Penalized Cox proportional hazards regressions with the elastic net penalty.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: Illumina HumanMethylation450 microarray platform
model.metadata["population"] = "adults"  # Paper: WHI, FHS-JHU, and LBC were the initial training cohorts.
model.metadata["journal"] = "Journal of the American Heart Association"
model.metadata["last_author"] = "José M. Ordovás"
model.metadata["n_features"] = 235
model.metadata["citations"] = 53
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o coefficients.csv https://raw.githubusercontent.com/bio-learn/biolearn/180852e2bab473303cb85da627178b1695ee9d86/biolearn/data/CVD_Westermann.csv")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
mask = df['CpGmarker'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'CoefficientTraining'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['CpGmarker'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['CoefficientTraining'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = 'sigmoid'
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Westerman, Kenneth, et al. "Epigenomic assessment of '
             'cardiovascular disease risk and interactions with traditional '
             'risk metrics." Journal of the American Heart Association 9.8 '
             '(2020): e015299.',
 'clock_name': 'cvdwesterman',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1161/JAHA.119.015299',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: 'sigmoid'
postprocess_dependencies: None
features: ['cg13077366', 'cg13679303', 'cg19214707', 'cg18593317', 'cg03463778', 'cg26626449', 'cg05630272', 'cg23522872', 'cg12588880', 'cg13074055', 'cg02655711', 'cg12624197', 'cg05843457', 'cg10963061', 'cg12121643', 'cg23613051',

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
